# Planet API Imagery Orders Workflow

The goal of this notebook is to download Planet imagery, that will be processed into tabulated NDVI data. We accomplish this by:

1. Initializing the variables and parameters of our query
2. Making our query and returning a list of Planet scene IDs
3. Sending order requests by year and month based on the obtained IDs
4. Pinging the API to source the download IDs by month and year
5. Downloading the imagery in .zip format.

## Declare imports and URL variables

In [ ]:
# location of API key and geojson creation funcions
sys.path.append("../utils")

# declare libraries
import json
import os
import sys
import time
from zipfile import ZipFile

import aoi_filter_maker as aoi
import geopandas as gpd
import requests
from planet import Planet, Session
from planet.auth import APIKeyAuth
from requests.adapters import HTTPAdapter
from requests.auth import HTTPBasicAuth
from urllib3.util.retry import Retry
import config

# Planet API urls
data_api_url = "https://api.planet.com/data/v1"
orders_api_url = 'https://api.planet.com/compute/ops/orders/v2' 

## Authorization and Initialization

1. Set the API key variables
2. Connect to the API
3. Initialize retry logic
4. Make the pagenation function

In [ ]:
# load api key into a local variable
planet_key = config.planet_api

# authentication
auth = HTTPBasicAuth(planet_key, "")

# check we are communicating with http properly
response = requests.get(data_api_url, auth=auth, timeout=999)
print(response)

<Response [200]>


In [ ]:
# setup
session = requests.Session()

# init retry logic
retries = Retry(
    total=5, # total retries
    backoff_factor=2, # increase the time before trying again by this much
    status_forcelist=[500, 502, 503, 504] # retry on these error codes
    )
session.mount("https://", HTTPAdapter(max_retries=retries)) # specify session to be Hypertext Transfer Protocol Secure

# authenticate with planet key
session.auth = (planet_key, "")

# check for proper connection
res = session.get(data_api_url)
res

<Response [200]>

In [ ]:
# make function for pagenation
def p(data):
    print(json.dumps(data, indent = 2))

### Load AOI filter

This is a set of polygons covering nearly every parcel.

In [ ]:
# init aoi filter format
geojson_custom_polys = {
        "type": "MultiPolygon", 
        "coordinates": []
    }

# read in polygons from file and convert to geojson
custom_polys = gpd.read_file('/capstone/wildfire_prep/josh/data-preparation/code/planet_tests/test_geojsons/geojson_io.geojson')
custom_polys = custom_polys.to_geo_dict()["features"]

# reformat for geometry filter
for i in range(len(custom_polys)):
    mid = custom_polys[i]["geometry"]["coordinates"]
    mid = aoi.convert_tuples_to_lists(mid)
    geojson_custom_polys["coordinates"].append(mid)


geojson_custom_polys


{'type': 'MultiPolygon',
 'coordinates': [[[[-119.56771809093945, 34.44322375197481],
    [-119.73663012502135, 34.52177301933281],
    [-119.75723860573646, 34.552024644848075],
    [-119.90608462520964, 34.551024517135744],
    [-119.90489964411037, 34.535879563688056],
    [-119.86632579935241, 34.52327022281368],
    [-119.85947402590858, 34.481262928640135],
    [-119.91775893735735, 34.493959254540115],
    [-119.97169787501309, 34.49950780993696],
    [-120.04003311442314, 34.49938823550242],
    [-120.0433217582663, 34.51569445187353],
    [-119.99096661878949, 34.53341344381214],
    [-120.00707291986618, 34.550001320207826],
    [-120.10958875337107, 34.52737999905587],
    [-120.10752419079324, 34.48718375628877],
    [-120.21187377884269, 34.48324690352918],
    [-120.21976463729123, 34.49543092328773],
    [-120.16185456343968, 34.53037153367963],
    [-120.18739857360272, 34.58401864493493],
    [-120.20972233046169, 34.57886190004078],
    [-120.22485665499316, 34.519192

## Declare functions
1. place_order()
    * Takes the list of scene IDs + any tool specifications and sends an order request to the API
2. poll_for_success()
    * Repeatedly queries the API for order status. Finishes execution once order is successful.

In [ ]:
def place_order(request, auth, retry_counter, index, month):
    order_url = None

    # make order request
    response = session.post(
        orders_api_url, # provide url to the orders API
        data = json.dumps(request), # pass our order request so it can be posted
        auth = auth, # our authentication function
        headers = headers, # provide metadata, specify to return json
        timeout = 999 # wait for this long for a response before timing out
        )


    # if there is a rate limiting problem
    if response.status_code == 429:
        while retry_counter < 5 and response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))  # set 60 secs for the waiting time
            print(f"Rate limit hit. Retrying after {retry_after} seconds...")

            time.sleep(retry_after) # wait for 60 secs
            retry_counter = retry_counter + 1 # increment the retry counter

            return place_order(request, auth, retry_counter) # try the function again
    
    elif response.status_code == 400:
        print(f"Status code: {response.status_code}")
        print(f"Status comments: {response.text}")
        print("Bad Request, cancelling this order...")
        print(f"Bad index: {index} in month: {month}")
        return None



    json_error_i = 0
    while json_error_i < 5:
        try:
            # get ids of scenes
            order_id = response.json()["id"]
            print(order_id)

            # construct the url of our order
            order_url = orders_api_url + '/' + order_id

            break

        # retry after errors
        except json.JSONDecodeError:
            print("JSON parsing failed, attempting again...")
            json_error_i += 1
        except Exception:
            print("Some other error occured. Attempting again...")
            json_error_i += 1
        
        # and exit if we retry too many times
        if json_error_i >= 5:
            print("Retry limit reached. Ordering failed in the place_order function.")
            return None

    
    return order_url

In [6]:
def poll_for_success(order_url, auth, num_loops = 999): 
    i = 0
    while(i < num_loops): 

        # iterate
        i += 1

        # get order request
        r = requests.get(order_url, auth = auth, timeout=999)
        response = r.json()

        # grab current state
        state = response["orders"][0]["state"]
        print(state)

        # compare it to a variety of end states and print it
        end_states = ["success", "failed", "partial"]
        if state in end_states:
            print(f"End State: {state}")
            break

        # wait 30 secs
        time.sleep(30)



## Make iteration variables

These are used to cycle through years and months in the following ordering workflow. 

In [8]:
month_nums = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "01"]
year_nums = ["2019", "2020", "2021", "2022", "2023"]

i = 0
for year in year_nums: 
    globals()[f"year_iter_{year}"] = [*[str(2019 + i)] * 12, str(2020 + i)]
    i += 1

## Ordering workflow

The full workflow that makes every order.

In [ ]:
"""REMEMBER TO SET CHUNKS AND YEAR ITER VARIABLES!!!!"""

area_of_interest = geojson_custom_polys
year = 2023
# added the index for which to contine from which the collection last errored

# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":

    # index = 0

    """
    0 = jan
    1 = feb
    2 = mar
    3 = apr
    4 = may
    5 = jun
    6 = jul
    7 = aug
    8 = sep
    9 = oct
    10 = nov
    11 = dec
    """

    # iterate through a given year
    for i in range(len(globals()[f'year_iter_{year}'])-1):

        j = 1 # iterator for names
        month = 1

        """
        Begin time count
        """
        t_before = time.time()


        """
        Define our filters
        """
        # define current geometry filter
        geometry_filter = {
            "type": "GeometryFilter", # set filter type as geometry
            "field_name": "geometry", # give json column containing geometry
            "config": area_of_interest # input json
        }


        # define date range, should be for a single year
        date_range_filter = {
            "type": "DateRangeFilter", 
            "field_name": "acquired",  # selects for when imagery was measured, not published
            "config": { 
                "gte": f"{globals()[f'year_iter_{year}'][i]}-{month_nums[i]}-01T00:00:00.000Z", # greater then equal to
                "lt":  f"{globals()[f'year_iter_{year}'][i+1]}-{month_nums[i+1]}-01T00:00:00.000Z" # less then
            }
        }
        print(f"gte: {globals()[f'year_iter_{year}'][i]}-{month_nums[i]}-01T00:00:00.000Z")
        print(f"lt: {globals()[f'year_iter_{year}'][i+1]}-{month_nums[i+1]}-01T00:00:00.000Z")


        # cloud filter
        cloud_cover_filter = {
            "type": "RangeFilter", # generalized filter type that takes range of values
            "field_name": "cloud_cover", # ask for cloud cover
            "config": {
                "lte": 0.01 # filter for all scenes w <1% cloud cover
            }
        }

        # specifies only scenes that have 4 band orthorectified imagery available, will error otherwise
        asset_filter = {
            "type": "AndFilter",
            "config": [
                {
                    "type": "AssetFilter",
                    "config": [
                        "ortho_analytic_4b"
                    ]
                }
            ]
        }


        # combine filters
        combined_filters = {
            "type": "AndFilter", # filter type for AND conditional
            "config": [geometry_filter, date_range_filter, cloud_cover_filter, asset_filter]
        }





        """
        Take our filters and make a search request
        """
        # defines package type
        # PSScene has orthorectified 8 and 4 band imagery with the udm2 file
        # you can take imagery from multiple types but we only need this one
        item_types = ["PSScene"]

        # feed our filters and package type selection into a filter dict
        search_request = {
            "item_types": item_types,
            "filter": combined_filters
        }

        # allow 5 errors/retries before timing out
        chunked_error_cnt = 0

        while chunked_error_cnt < 5:
            try: 
                
                # sessions version - maintains connection over requests
                # allows retry logic, but may cause some unexpected behaviors, trying it for now
                search_result = \
                    session.post( # post sends our request to https api
                        "https://api.planet.com/data/v1/quick-search",
                        auth = HTTPBasicAuth(planet_key, ''),  # we give it our key
                        json = search_request, # and our request dict
                        timeout=999
                    )
                
                break
            except requests.exceptions.ChunkedEncodingError:
                if chunked_error_cnt < 5: 
                    print("Chunked encoding error on this chunk. Atempting again...")
                else:
                    print(f"Max retries reached on chunked error handler. Restart ordering workflow on month {i} to try again...")
                
                chunked_error_cnt += 1


        # remove stray scenes without the ortho_analytic_4b_sr asset that weren't caught in the asset filter
        new_list = []
        error_cnt = 0

        while error_cnt <= 5:
            try: 

                # get our search result and pass it to a variable for comparison
                edit_this = search_result.json()
                baseline_search = edit_this

                # for all scenes in the query
                for scene in enumerate(edit_this["features"]):

                    # if it contains the 4 band orthrorectified,
                    if "ortho_analytic_4b_sr" in scene[1]["assets"]:
                        # then keep it in the query
                        new_list.append(scene[1])
                    
                # if scenes have been removed, then make that fact know, and return the new list
                if len(new_list) != len(edit_this["features"]):
                    edit_this["features"] = new_list
                    print(f"Index has scenes without ortho_analytic_4b_sr. Removed {len(baseline_search['features']) - len(new_list)} scenes...")
                
                break

            # add retry logic
            except json.JSONDecodeError:
                if error_cnt < 5:
                    print("JSON parsing error in the stray scene removal step. Retrying...")
                else:
                    print("JSON parsing error in the stray scene removal step met max retries. This chunk may fail...")

                error_cnt += 1

            except Exception as e:
                if error_cnt < 5:
                    print("Some other error occured. Attempting again...")
                else:
                    print(f"Ordering failed in the stray scene removal due to some error, and reached max retries: {e}")

                error_cnt += 1
            
            if error_cnt > 5:
                break



        """
        Crunch all IDs into a list
        """
        ids = [feature['id'] for feature in edit_this["features"]]




        """
        Make our order request

        1. Setup our order json
        2. Setup our tools
        """
        # set content type to json
        headers = {"content-type": "application/json"}

        # init order parameters, this one has the 4 band
        product = [
            { 
                "item_ids": ids, 
                "item_type": "PSScene", # specify planetscope imagery
                "product_bundle": "analytic_sr_udm2", # ortho 4 band surface reflectance
            }
        ]


        # init clip to sb county
        clip =  {
            "clip": {
                "aoi": area_of_interest # our aoi filter from earlier
            }
        }

        # reprojects to CRS California Albers at point of order
        reproject = {
            "reproject": {
                "projection": "EPSG:3310",
                "kernel": "bilinear"
            }
        }


        # make name
        order_name = f"custom_polys_month_{month_nums[i]}_year_{year}_download_{j}_first_run"

        # create request json
        tool_request = { 
            "name": order_name, 
            "products": product, # ids and satellite specifications
            "tools": [clip, reproject], # add tools
            "delivery": {
                "single_archive": True, # specifies to archive all files together in a single bundle
                "archive_type": "zip" # specifies to make the order in zip format
                }
        }



        """
        Send in the order
        """
        
        retry_counter = 0

        # call the function
        order_url = place_order(tool_request, auth, retry_counter, j-1, month = j)
        print("\n")


        """
        Wait before sending in next order
        """

        # wait before making next order as to not overload the api
        time.sleep(0.5)

        # and increment the order labeler
        j = j + 1



    # now we can poll for success
    poll_for_success(orders_api_url, auth)


    """
    End time count and output total time PARTIALLY DEPRECATED, MOVED TO OUTSIDE THE FOR LOOP
    """

    # get end time
    t_after = time.time()
    # get difference for total execution time
    custom_polys_exec_t = t_after - t_before
    
    print(f"Total execution time for {order_name}: {custom_polys_exec_t} seconds")
        

#________________End of code chunk___________________#

    print("Cell executed!")
else:
    print("Execution canceled.")

gte: 2023-01-01T00:00:00.000Z
lt: 2023-02-01T00:00:00.000Z
d23534f5-b17a-4dfb-a1e3-921c94c65e53


gte: 2023-02-01T00:00:00.000Z
lt: 2023-03-01T00:00:00.000Z
fb7a4040-7338-4dec-8d2e-95cdf4b1b3a0


gte: 2023-03-01T00:00:00.000Z
lt: 2023-04-01T00:00:00.000Z
53bf83ed-ba19-4f4e-aa79-d034d5d396d7


gte: 2023-04-01T00:00:00.000Z
lt: 2023-05-01T00:00:00.000Z
7ab54c92-7bc3-4d9f-bb3b-6f8665ec21cf


gte: 2023-05-01T00:00:00.000Z
lt: 2023-06-01T00:00:00.000Z
9da791dc-1fce-4269-88ab-039720994cf1


gte: 2023-06-01T00:00:00.000Z
lt: 2023-07-01T00:00:00.000Z
74ec5d00-2bf7-4b71-b69a-76fc944bb3fd


gte: 2023-07-01T00:00:00.000Z
lt: 2023-08-01T00:00:00.000Z
bebe840b-a806-4599-aaa1-1052ee8215d5


gte: 2023-08-01T00:00:00.000Z
lt: 2023-09-01T00:00:00.000Z
6524c5bb-ca01-4c6c-8cf0-b106a156c3a7


gte: 2023-09-01T00:00:00.000Z
lt: 2023-10-01T00:00:00.000Z
89752200-9099-4efd-8522-1c9c52944bf9


gte: 2023-10-01T00:00:00.000Z
lt: 2023-11-01T00:00:00.000Z
079043c9-6268-4422-a0d2-c26bdecd6c75


gte: 2023-11-01T00:0

KeyboardInterrupt: 

## Get all previous orders

This chunk gets a list of every previously made order. We will subset this list to retrieve the orders just made.

In [ ]:
# ping planet
r = requests.get(orders_api_url, auth = auth)

# turn response into a json/dict
order_response = r.json()

# isolate state of most recent order
order_results = order_response["orders"][0]["state"]



all_scenes = []

# grab metadata for every order the current account has ever made
first_scenes = [[squid["name"], squid["id"], squid["created_on"]] for squid in order_response["orders"]]
all_scenes.extend(first_scenes)

json_error_counter = 0
i = 0

# wrapper to handle json decode errors
def json_decode_error_handler(func, args):
    json_error_counter = 0
    result = None

    while json_error_counter < 10:

        # try some code
        try:
            result = func(*args)
            break
        # and retry if it fails, up to 10 times
        except json.JSONDecodeError:
            json_error_counter = json_error_counter + 1
    
    if json_error_counter < 10:
        print(f"Completed successfully. Json decode errors: {json_error_counter}...") 
    elif json_error_counter >= 10:
        print("At max Json decode errors. Completed unsuccessfully. Returning Null...")

    return result


# pagenate through scenes
# lists of scenes come in groups of 250, with a link to ping for the next 250 afterward
def scene_pagenation(order_response, scenes, depth): 
    scenes_iter = []
    retry_counter = 0

    try:
        for i in range(depth):
            # check if there is a link to fetch more scenes
            if "_links" in order_response and "next" in order_response["_links"]:
                next_link = order_response["_links"]["next"] # fetch it
                next_scenes_ping = session.get(next_link) # then send it into the api to get the next group of scenes
            else:
                print("No more pages available. Ending pagination.")
                break  # Prevent error when no next page

            # sleep to avoid overloading the api with requests
            time.sleep(0.5)

            # if there is an error, print the error code and message
            if next_scenes_ping.status_code != 200:
                print(f"Error {next_scenes_ping.status_code}: {next_scenes_ping.text}")

            # otherwise, just continue the function
            else:
                pag_json = next_scenes_ping.json() # convert the curr 250 to json format

            # grab certain metadata from the query
            next_scenes = [
                [
                    squid["name"], 
                    squid["id"], 
                    squid["created_on"], 
                    squid["products"][0]["item_ids"][0]
                ] 
                    for squid in pag_json["orders"]
                ]

            # append that metadata to a list
            scenes_iter.append(next_scenes)

            # and reassign the json
            order_response = pag_json
            
    except Exception as e:
            print(f"Error during pagination at depth {i}: {e}. Retrying...")

            time.sleep(2 ** retry_counter)  # Exponential backoff
            retry_counter += 1

            # retry the function after the backoff
            scene_pagenation(order_response, scenes, depth)

            

    # at end of function, concat all scenes into a master list
    for chunk in scenes_iter:
        scenes.extend(chunk)

    return scenes

# call the pagenation function, wrapped in the error handling wrapper
all_scenes = json_decode_error_handler(
    scene_pagenation, [order_response, all_scenes, 9999]
)





No more pages available. Ending pagination.
Completed successfully. Json decode errors: 0...


#### Construct updated iteration variables

In [16]:
month_sea = ["month_12", "month_11", "month_10", "month_09", "month_08", "month_07", "month_06", "month_05", "month_04", "month_03", "month_02", "month_01"]
month_name = ["dec", "nov", "oct", "sep", "aug", "jul", "jun", "may", "apr", "mar", "feb", "jan"]
year_sea = ["2019", "2020", "2021", "2022", "2023"]


#### Print all scenes

In [100]:
all_scenes

[['custom_polys_month_12_year_2023_download_1_first_run',
  'dee134e2-2caf-4247-ba8a-dfaf1da1df83',
  '2025-05-29T10:06:54.392227Z'],
 ['custom_polys_month_11_year_2023_download_1_first_run',
  'f64cab73-e976-40a7-9ba3-f9267267d1ec',
  '2025-05-29T10:06:50.316674Z'],
 ['custom_polys_month_10_year_2023_download_1_first_run',
  '079043c9-6268-4422-a0d2-c26bdecd6c75',
  '2025-05-29T10:06:46.513094Z'],
 ['custom_polys_month_09_year_2023_download_1_first_run',
  '89752200-9099-4efd-8522-1c9c52944bf9',
  '2025-05-29T10:06:42.761767Z'],
 ['custom_polys_month_08_year_2023_download_1_first_run',
  '6524c5bb-ca01-4c6c-8cf0-b106a156c3a7',
  '2025-05-29T10:06:39.1986Z'],
 ['custom_polys_month_07_year_2023_download_1_first_run',
  'bebe840b-a806-4599-aaa1-1052ee8215d5',
  '2025-05-29T10:06:35.448303Z'],
 ['custom_polys_month_06_year_2023_download_1_first_run',
  '74ec5d00-2bf7-4b71-b69a-76fc944bb3fd',
  '2025-05-29T10:06:31.679521Z'],
 ['custom_polys_month_05_year_2023_download_1_first_run',
  '9da

## Subset the orders

We can subset the orders if not by name, than by other metadata such as date of creation or selecting the ID directly. Both for loops and list comprehension.

**Note**: An error was made where the orders for 2020 were listed as being in 2021.

In [ ]:
# method 1 ---- for loops

# declare the empty lists
list_scenes = []
list_2020 = []
list_2021 = []
list_2022 = []
list_2023 = []

"""
# and run through every month iterable vars
for m_search, m_name in zip(month_sea, month_name):

    # within it, run through every scene
    for scene in all_scenes:
    
        # subset by substrings in the metadata
        if m_search in scene[0] and "run" in scene[0] and "dry_run" not in scene[0] and "custom_polys" in scene[0]:
        
            # and do it again for each year
            if "2021" not in scene[0] and "2025-05-29" in scene[2]:
                list_2020.append(scene[1]) # then append it to their respective empty lists
            
            # do the same for the rest
            elif "2021" in scene[0]:
                list_2021.append(scene[1])
            if "2022" in scene[0]:
                print(f"put in 2022: {scene[0]}")
                list_2022.append(scene[1])
            elif "2023" in scene[0]:
                print(f"put in 2023: {scene[0]}")
                list_2023.append(scene[1])
"""

put in 2023: custom_polys_month_12_year_2023_download_1_first_run
put in 2022: custom_polys_month_12_year_2022_download_1_first_run
put in 2023: custom_polys_month_11_year_2023_download_1_first_run
put in 2022: custom_polys_month_11_year_2022_download_1_first_run
put in 2023: custom_polys_month_10_year_2023_download_1_first_run
put in 2022: custom_polys_month_10_year_2022_download_1_first_run
put in 2023: custom_polys_month_09_year_2023_download_1_first_run
put in 2022: custom_polys_month_09_year_2022_download_1_first_run
put in 2023: custom_polys_month_08_year_2023_download_1_first_run
put in 2022: custom_polys_month_08_year_2022_download_1_first_run
put in 2023: custom_polys_month_07_year_2023_download_1_first_run
put in 2022: custom_polys_month_07_year_2022_download_1_first_run
put in 2023: custom_polys_month_06_year_2023_download_1_first_run
put in 2022: custom_polys_month_06_year_2022_download_1_first_run
put in 2023: custom_polys_month_05_year_2023_download_1_first_run
put in 202

In [ ]:
# method 2 ---- list comprehension

# for every scene in the scenes list, subset it for a given year
list_2021 = [scene[1] for scene in all_scenes if "2021" in scene[0] and "custom_polys" in scene[0]]
list_2022 = [scene[1] for scene in all_scenes if "2022" in scene[0] and "custom_polys" in scene[0]]
list_2023 = [scene[1] for scene in all_scenes if "2023" in scene[0] and "custom_polys" in scene[0]]


list_2021

['d0f14bf1-5524-41bc-8d1f-44296a64f582',
 'd17ead9e-0528-4ee8-b5ab-0f9c2914f22f',
 '0ed6e7c1-b758-4e89-ab9d-d017277b3c47',
 'eb01fd17-d70a-4a5a-8bb7-bb626b5d455c',
 '33b1590f-e379-4fed-946f-9298099f5d89',
 'f9fddd31-eaf6-4abc-a5b4-fad24b4d403c',
 'da4e96a4-2bc9-42bb-8efa-3a5617b364f8',
 '4c5ed53c-00dc-451d-840d-c4778f29b5a4',
 'fd0c98c1-28b0-430d-a8fd-902ab6cdb536',
 '07a86049-271f-4491-9213-7242b6473316',
 '65bf782a-845a-400c-8e01-d9fe9fdffc05',
 'e9fa0e69-6b85-43f0-93b6-15fc3b59771c']

#### Dynamically stick subsetted lists of orders in a variable

Variable is named by month and year.

In [ ]:
for m_search, m_name in zip(month_sea, month_name):

    append_list = []

    for scene in all_scenes:

        # subset by substring
        if m_search in scene[0] and "run" in scene[0] and "dry_run" not in scene[0] and "custom_polys" in scene[0]:
            append_list.append(scene) # and append to the master list

    # declare the variable by month and year
    globals()[f"all_{m_name}_{year_sea[3]}_prelim"] = append_list
    

month_12/dec
month_11/nov
month_10/oct
month_09/sep
month_08/aug
month_07/jul
month_06/jun
month_05/may
month_04/apr
month_03/mar
month_02/feb
month_01/jan


In [54]:
for name in month_name:
    print(name)

dec
nov
oct
sep
aug
jul
jun
may
apr
mar
feb
jan


## Download Bundles...
#### ...and unzip imagery

In [ ]:
# input what years your want to download
download_lists = [list_2022, list_2023]
download_years = [2022, 2023]

# ping the planet api again
pl = Planet(session=Session(auth=APIKeyAuth(key=planet_key)))

# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":
            
    except_counter = 0
    sleep_counter = 1

    # iterate through your chosen years/lists
    for download_list, download_year in zip(download_lists, download_years):
    
        # for every year, run through every scene and their associated month
        for m_name, scene in zip(month_name, download_list):

            except_counter = 0
            sleep_counter = 1

            # make the order
            while True:
                try: 
                    pl.orders.download_order(
                        scene, # get order id
                        overwrite = False, # don't overwrite files that already exist
                        directory = f"/data/wildfire_prep/full_planet/raw_zip/{download_year}/{m_name}" # set directory to our data folder in workbench-2
                        )
                    break

                # retry logic
                except Exception as e:
                    if except_counter < 5:
                        print(f"Error at {scene}: {e}. Retrying...")
                        except_counter += 1 
                        
                        time.sleep(sleep_counter)
                        sleep_counter = sleep_counter * 2
                        print(f"Waiting for {sleep_counter} seconds before attempting again...")

    print("Cell executed!")
    
else:
    print("Execution canceled.")

Cell executed!


In [ ]:
extract_path = "/data/wildfire_prep/full_planet/raw_zip" # location of zips
root_export_path = "/data/wildfire_prep/full_planet/unzipped" # output path

# years to be unzipped
years = [2021,2022,2023]

# list of filepaths to be unzipped
unzip_list = []

# run through every month in every year
for year in years:
    for month in month_name: 

        # sift through the file structures in the 
        for root,dirs,files in os.walk(extract_path, topdown=True):

            # and select the zip file in every appropriate directory
            if "output.zip" in files and f'{year}/' in root and f'{month}/' in root:
                print(f"{root[32:]}/{files[1]}")
                unzip_list.append(f"{root}/{files[1]}") # append the filepath to the list
unzip_list


In [ ]:

# safety check, only run cell if you specifically say "yes"

years = [2023]

response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":

    for year in years:
        for month in month_name: 

            # run through filepaths
            for root,dirs,files in os.walk(extract_path, topdown=True):
                
                # with the zip in a given month/year
                if "output.zip" in files and f'{year}/' in root and f'{month}/' in root:
                    with ZipFile(f"{root}/output.zip", 'r') as zip: 


                        print(f"Unzipping {root[32:]}/{files[1]} to unzipped/{year}/{month}")
                        print('Extracting all the files now...') 

                        # extract the zips to your chosen filepath
                        zip.extractall(path = f"{root_export_path}/{year}/{month}") 
                        
                        print('Done!') 
                        print("---------------------------\n")



    print("Cell executed!")
else:
    print("Execution canceled.")

Unzipping raw_zip/2023/dec/dee134e2-2caf-4247-ba8a-dfaf1da1df83/output.zip to unzipped/2023/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023/nov/f64cab73-e976-40a7-9ba3-f9267267d1ec/output.zip to unzipped/2023/nov
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023/oct/079043c9-6268-4422-a0d2-c26bdecd6c75/output.zip to unzipped/2023/oct
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023/sep/89752200-9099-4efd-8522-1c9c52944bf9/output.zip to unzipped/2023/sep
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023/aug/6524c5bb-ca01-4c6c-8cf0-b106a156c3a7/output.zip to unzipped/2023/aug
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023/jul/bebe840b-a806-4599-aaa1-1052ee8215d5/output.zip to unzipped/2023/jul
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2023